In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def parse_opennmt_log(log_file_path):
    """
    Phân tích file log của OpenNMT-py để trích xuất các chỉ số huấn luyện và kiểm tra.
    """
    # Các pattern regex để tìm các dòng chứa thông tin cần thiết
    train_pattern = re.compile(r"Step (\d+)/\d+; acc: ([\d.]+); ppl: ([\d.]+); xent: ([\d.]+);")
    valid_ppl_pattern = re.compile(r"Validation perplexity: ([\d.]+)")
    valid_acc_pattern = re.compile(r"Validation accuracy: ([\d.]+)")

    # Khởi tạo các danh sách để lưu dữ liệu
    data = {
        "step": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    
    last_step = 0
    
    with open(log_file_path, 'r', encoding='utf-8') as f:
        # Chỉ lấy phần log của lần chạy cuối cùng
        # Tìm dòng "Building model..." cuối cùng
        log_content = f.read()
        last_run_start = log_content.rfind("Building model...")
        if last_run_start == -1:
             last_run_start = 0 # Nếu không tìm thấy, bắt đầu từ đầu
        
        log_lines = log_content[last_run_start:].splitlines()

        for line in log_lines:
            train_match = train_pattern.search(line)
            valid_ppl_match = valid_ppl_pattern.search(line)
            valid_acc_match = valid_acc_pattern.search(line)

            if train_match:
                step = int(train_match.group(1))
                acc = float(train_match.group(2))
                # xent (cross-entropy) là loss thực tế
                xent = float(train_match.group(4)) 
                
                # Lưu chỉ số train
                data["step"].append(step)
                data["train_loss"].append(xent)
                data["train_acc"].append(acc)
                
                # Cập nhật bước cuối cùng
                last_step = step

            elif valid_ppl_match:
                # Dòng validation loss (perplexity) xuất hiện sau dòng step
                # Chúng ta sẽ gắn nó với step cuối cùng thấy được
                val_ppl = float(valid_ppl_match.group(1))
                # Tìm index của step này trong dataframe
                try:
                    idx = data["step"].index(last_step)
                    # Gán giá trị val_loss vào đúng hàng
                    if len(data["val_loss"]) <= idx:
                         # Điền các giá trị NaN vào các hàng trước đó
                         data["val_loss"].extend([float('nan')] * (idx - len(data["val_loss"]) + 1))
                    data["val_loss"][idx] = val_ppl
                except ValueError:
                    continue

            elif valid_acc_match:
                # Tương tự với validation accuracy
                val_acc = float(valid_acc_match.group(1))
                try:
                    idx = data["step"].index(last_step)
                    if len(data["val_acc"]) <= idx:
                         data["val_acc"].extend([float('nan')] * (idx - len(data["val_acc"]) + 1))
                    data["val_acc"][idx] = val_acc
                except ValueError:
                    continue

    # Tạo DataFrame từ dictionary
    df = pd.DataFrame(data)
    # Loại bỏ các hàng không có giá trị step (dữ liệu rác)
    df.dropna(subset=['step'], inplace=True)
    # Chuyển đổi cột step sang kiểu integer
    df['step'] = df['step'].astype(int)
    # Sắp xếp lại theo step
    df.sort_values('step', inplace=True)
    
    # Sử dụng forward-fill để điền các giá trị validation bị thiếu
    df['val_loss'].ffill(inplace=True)
    df['val_acc'].ffill(inplace=True)

    return df

def plot_training_results(df):
    """
    Vẽ biểu đồ loss và accuracy từ DataFrame.
    """
    # Thiết lập 2 biểu đồ con trên cùng một hình
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
    
    # --- Biểu đồ 1: Loss ---
    # Chú ý: Training loss là 'xent', Validation loss là 'ppl'. Chúng có thang đo khác nhau
    # nhưng vẫn thể hiện xu hướng chung.
    ax1.plot(df['step'], df['train_loss'], label='Training Loss (Cross-Entropy)', color='dodgerblue', alpha=0.8)
    # Lấy dữ liệu validation không phải NaN để vẽ điểm
    val_loss_points = df.dropna(subset=['val_loss'])
    ax1.plot(df['step'], df['val_loss'], label='Validation Loss (Perplexity)', color='orangered', linestyle='--')
    ax1.scatter(val_loss_points['step'], val_loss_points['val_loss'], color='red', s=15, zorder=5) # Đánh dấu các điểm validation

    ax1.set_ylabel("Giá trị Loss")
    ax1.set_title("Training & Validation Loss qua các bước")
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    # --- Biểu đồ 2: Accuracy ---
    ax2.plot(df['step'], df['train_acc'], label='Training Accuracy (%)', color='dodgerblue', alpha=0.8)
    val_acc_points = df.dropna(subset=['val_acc'])
    ax2.plot(df['step'], df['val_acc'], label='Validation Accuracy (%)', color='orangered', linestyle='--')
    ax2.scatter(val_acc_points['step'], val_acc_points['val_acc'], color='red', s=15, zorder=5)

    ax2.set_xlabel("Training Steps")
    ax2.set_ylabel("Độ chính xác (%)")
    ax2.set_title("Training & Validation Accuracy qua các bước")
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.6)

    # Định dạng trục x cho dễ đọc
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: format(int(x), ',')))
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

# --- Main ---
if __name__ == "__main__":
    # Thay 'training_log.txt' bằng tên file log thực tế của bạn
    log_file = '/home/leloc/Document/USTH/Thesis/Machine_translation-Thai-Viet-/nllb-200-3.3B-onmt.log' 
    
    try:
        training_df = parse_opennmt_log(log_file)
        print("Đã phân tích xong log. Dữ liệu thu được:")
        print(training_df.head())
        print(f"\nTổng số điểm dữ liệu huấn luyện: {len(training_df)}")
        print(f"Tổng số điểm dữ liệu kiểm tra: {training_df['val_loss'].notna().sum()}")
        
        plot_training_results(training_df)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{log_file}'. Vui lòng đảm bảo file log nằm trong cùng thư mục.")